# UMSS on MASSVIS

Do our two UMSS stages give the paper's results?

- **S1** predicts three temporal saliency maps (0–0.5 s, 0.5–3 s, 3–5 s). Compared with Table 2.
- **S2** samples scanpaths from those maps. Compared with Table 1.

Each table shows the paper, the authors' released outputs and ours, all scored with the same evaluation
(the authors' code in `official_repository/`). The released outputs check the evaluation: they should give the
paper's numbers. Our maps come from one of the two MD-EAM weights retrained in this repository, chosen with
`WEIGHTS` below: `data/ours_last_epoch/npz/` (new, last epoch) or `data/ours/npz/` (old), exported with
`run.sh export`. S1 scores the chart-size PNGs the export
writes in `png/` with the authors' `predict_and_save_md`; exports without them fall back to the grid maps.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'models/umss').is_dir())
sys.path.insert(0, str(ROOT))

from reproductions.umss_massvis import replication as R

# Which retrained weights' maps to score: 'last_epoch' (md_eam_last_epoch.hdf5) or 'old' (md_eam.hdf5).
WEIGHTS = 'last_epoch'
MAPS = {'last_epoch': R.OUR_MAPS_LAST_EPOCH, 'old': R.OUR_MAPS}[WEIGHTS]

RESULTS = R.EXPERIMENT / 'results'
RESULTS.mkdir(exist_ok=True)

## Data

Wang, Bâce and Bulling, *Scanpath Prediction on Information Visualisations*, TVCG 2023.
The evaluation uses the 66 charts of the published split (every sixth of the 393 MASSVIS charts).

The first time, restore the deposit with `sync_official_repository.py` and run `R.download()` to unzip the
images and released outputs and fetch the MASSVIS fixations and element labels.

In [ ]:
# R.download()
all_names, names, paths = R.images()
print('MASSVIS charts:', len(all_names), '| evaluation charts:', len(names))

## S1: saliency maps (Table 2)

NSS, CC, KL and SIM of the 0.5–3 s and 3–5 s maps against all human fixations up to 3 s and 5 s
(Gaussian sigma 19 px), as in the authors' evaluation. Higher is better except for KL.

In [ ]:
s1 = R.s1(MAPS)
s1.to_csv(RESULTS / f's1_{WEIGHTS}.csv')
display(s1.round(3))

## S2: scanpaths (Table 1)

40 scanpaths per chart from our maps with `UMSS.run(n_fix=None)`, the official UMSS-S2 sampler (run r uses seed r).
They are compared with the viewers' first 5 seconds by Sequence Score, scaled TDE, DTW and MultiMatch.
Higher is better except for DTW. ScanMatch needs MATLAB and is not computed.

This step takes about an hour on 14 cores: the official sampler filters a full-size Gaussian per fixation.

In [ ]:
s2 = R.s2(MAPS)
s2.to_csv(RESULTS / f's2_{WEIGHTS}.csv')
display(s2.round(3))

## Notes

- The paper's 5 s human scanpaths and accumulated human maps are not in the deposit; they are rebuilt from
  MASSVIS with the rules of the authors' code. The released scanpaths do not give back the paper's
  Sequence Score and DTW best, which depend on that rebuild.
- Our maps come from the old checkpoint retrained with the authors' recipe. A second retraining that keeps the
  last epoch (`md_eam_last_epoch.hdf5`) gives the same Table 2 scores within 0.01, so ours being above the paper
  at 3 s is not from choosing the epoch on these charts; the published recipe does not give the paper's numbers.
- S2 is random: another seed gives slightly different numbers.